In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler

In [ ]:
# --- Bước 1: Đọc dữ liệu ---
data = pd.read_csv('data.csv', parse_dates=['date'], index_col='date')
data = data.sort_index()

# --- Bước 2: Tiền xử lý dữ liệu ---
# Sử dụng MinMaxScaler để chuẩn hóa cột 'value'
scaler = MinMaxScaler(feature_range=(0, 1))
data['value_scaled'] = scaler.fit_transform(data[['value']])

# Tạo các đặc trưng lag. Ở đây, dùng 3 ngày trước để dự báo ngày kế tiếp
def create_lag_features(df, lags=3):
    df_features = pd.DataFrame()
    for lag in range(1, lags + 1):
        df_features[f'lag_{lag}'] = df['value_scaled'].shift(lag)
    # Giá trị mục tiêu: giá trị hiện tại
    df_features['target'] = df['value_scaled']
    return df_features.dropna()

lags = 3
df_features = create_lag_features(data, lags)

# Tách đặc trưng (X) và nhãn (y)
X = df_features.drop('target', axis=1).values
y = df_features['target'].values

# Chia dữ liệu thành tập huấn luyện và tập kiểm tra (80% - 20%)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=False)

# --- Bước 3: Xây dựng và huấn luyện mô hình Random Forest ---
model = RandomForestRegressor(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

# Dự báo trên tập kiểm tra
y_pred = model.predict(X_test)

# Tính toán lỗi dự báo
mse = mean_squared_error(y_test, y_pred)
print(f"MSE: {mse:.4f}")

# Chuyển đổi dự báo về giá trị ban đầu
y_test_inv = scaler.inverse_transform(y_test.reshape(-1, 1))
y_pred_inv = scaler.inverse_transform(y_pred.reshape(-1, 1))

# --- Bước 4: Trực quan hóa kết quả ---
plt.figure(figsize=(8, 5))
plt.plot(y_test_inv, label='Giá trị thực', marker='o')
plt.plot(y_pred_inv, label='Dự báo', marker='x', linestyle='--')
plt.title('Dự báo chuỗi thời gian với Random Forest')
plt.xlabel('Mẫu')
plt.ylabel('Giá trị')
plt.legend()
plt.show()

4. Giải thích chi tiết:
Tiền xử lý dữ liệu:

Dữ liệu được đọc từ data.csv và sắp xếp theo ngày.

Cột value được chuẩn hóa về khoảng [0,1] bằng MinMaxScaler.

Hàm create_lag_features tạo ra các đặc trưng lag (ở đây là 3 lag) để làm đầu vào dự báo giá trị hiện tại.

Huấn luyện mô hình:

Dữ liệu được chia theo thứ tự thời gian (không xáo trộn) để đảm bảo tính tuần tự cho bài toán chuỗi thời gian.

Sử dụng RandomForestRegressor với 100 cây (n_estimators=100).

Mô hình được huấn luyện trên tập huấn luyện và sau đó dự báo trên tập kiểm tra.

Đánh giá và trực quan hóa:

Tính toán MSE để đánh giá hiệu năng mô hình.

Dự báo được chuyển đổi về giá trị ban đầu và vẽ biểu đồ so sánh giữa giá trị thực và giá trị dự báo.